# Konu 9 — Ölçek: toplu üretim

Hücreleri **yukarıdan aşağı, sırayla** çalıştır: hücreye tıkla, **Shift + Enter**.
Her hücre bir öncekinin oluşturduğu değişkeni kullanır; atlarsan hata alırsın.

Konu 8'de görselleri tek tek ürettik. Bugün bir **şablon** ve birkaç listeyle bir görsel **ızgarası** üretiyoruz:
arayüzde tek tek yapılacak iş, kodda birkaç hücre.

Isınma, Adım 1 ve Adım 2 anahtarsız. Adım 3'ten sonrası `anahtar.txt` ister (Konu 2'deki dosya). Defterin tamamı 12 görsel
(Bonus ile en fazla 17) üretir ve 13 kez görsel anlayan bir modele soru sorar; ücretsiz günlük kotanın küçük bir kısmı.

## Isınma — Kaç kombinasyon?

Bir kafe afişi için üç stil ve dört renk paleti deneyeceğiz. Her stil her renkle bir kez: **kombinasyon**.

**Tahmin et, sohbete yaz:** slayttaki gardırop örneğini hatırla. 3 stil ve 4 renk kaç görsel eder? 5 stil ve 6 renk?

Önce yalnızca adları yazdırıyoruz. **İç içe döngü:** dıştaki döngü bir stil seçer; içteki döngü o stil için
bütün renkleri gezer. Sonra dıştaki döngü bir sonraki stile geçer.

In [ ]:
stiller = ["suluboya", "vektor", "linol"]
renkler = ["adacayi", "gece", "pastel", "toprak"]

for stil in stiller:
    for renk in renkler:
        print(stil, renk)

12 satır: her stil dört kez, her renk üç kez. `print` satırı **iki girinti** içeride: bir girinti `for stil`'in, ikincisi
`for renk`'in içinde olduğunu gösterir.

Satırları tek tek saymak yerine çarpabiliriz:

In [ ]:
print(len(stiller) * len(renkler))

5 stil ve 6 renk: 5 × 6 = 30. Elimizdeki 3 × 4'e bir renk eklersen 3 × 5 = 15; üstüne iki stil daha eklersen 5 × 5 = 25.
Liste büyüdükçe görsel sayısı **çarpılarak** büyür; kota da öyle.

## Adım 1 — Şablon ve kartlar

Dosya adı için kısa ve Türkçe karaktersiz adlar (`suluboya`) kullanıyoruz; modele ise İngilizce açıklamalar
(`watercolor illustration`) gidecek. İkisini **sözlükte** eşliyoruz (Konu 1): anahtar kısa ad, değer yönerge parçası.
`ozne` bütün görsellerde aynı: şablonun sabit kısmı. `brif` kendi notumuz (günlüğe yazılacak).

In [ ]:
ozne = "a ceramic coffee cup on a wooden desk next to an open notebook, soft morning light"
brif = "sessiz bir çalışma kafesi için afiş"

stil_yonergeleri = {
    "suluboya": "watercolor illustration",
    "vektor": "flat vector poster",
    "linol": "linocut print",
}
renk_yonergeleri = {
    "adacayi": "sage green and beige palette",
    "gece": "deep navy and warm amber palette",
    "pastel": "soft pastel palette",
    "toprak": "terracotta and olive palette",
}
print(stil_yonergeleri["linol"])

Her kombinasyonun bütün bilgisini bir **kart** içinde tutacağız: bir sözlük. Önce tek bir kartı elle yazalım:

In [ ]:
kart = {
    "stil": "suluboya",
    "renk": "gece",
    "dosya": "suluboya_gece.jpg",
    "yonerge": ozne + ", watercolor illustration, deep navy and warm amber palette",
    "tohum": 42,
}
print(kart["dosya"])

Şimdi aynı kartı her kombinasyon için döngüyle kuruyoruz. Önce iki yönerge parçasını sözlükten okuyup ayrı değişkenlere
alıyoruz; sonra kartı kurup `kartlar` listesine ekliyoruz. `"durum": "denenmedi"` başlangıç değeri: üretim sırasında
`"uretildi"` ya da `"olmadi"` olacak. Listeleri ya da sözlükleri değiştirirsen bu hücreyi yeniden çalıştır.

In [ ]:
kartlar = []
for stil in stiller:
    for renk in renkler:
        stil_metni = stil_yonergeleri[stil]
        renk_metni = renk_yonergeleri[renk]
        kart = {
            "stil": stil,
            "renk": renk,
            "dosya": f"{stil}_{renk}.jpg",
            "yonerge": f"{ozne}, {stil_metni}, {renk_metni}",
            "tohum": 42,
            "model": "flux-1-schnell",
            "brif": brif,
            "durum": "denenmedi",
        }
        kartlar.append(kart)

print(len(kartlar))

12 kart: bir liste, içinde 12 sözlük. İlk karta bakalım:

In [ ]:
ilk_kart = kartlar[0]
print(ilk_kart["dosya"])
print(ilk_kart["yonerge"])

Bu bir **şablon**: özne ve tohum sabit, stil ve renk düzenli olarak değişiyor. Konu 7'deki kontrollü deneyin iki eksenli hâli.

## Adım 2 — Kota ve güvenlik sınırı

Konu 2'den hatırla: Cloudflare kotayı **neuron** ile sayar; günde 10.000 neuron ücretsiz. Bir görsel (1024 × 1024, Konu 7'deki
gibi 4 adım) yaklaşık 58 neuron tutar. Hesap şöyle: Cloudflare 1024'lük görseli 512 × 512'lik 4 kare sayar; her kare 4,8,
her adım 9,6 neuron: 4 × 4,8 + 4 × 9,6 ≈ 58. Günde kabaca 170 görsel eder.

Döngüye girmeden önce **hesap yapıyoruz** ve bir **sınır** koyuyoruz: bir çalıştırmada en fazla kaç görsel üretilecek.
`MAKS_URETIM` büyük harfle yazılmış bir değişken: Python'da "bunu değiştirme, sabit" demenin geleneği.

In [ ]:
MAKS_URETIM = 12
GORSEL_BASI_NEURON = 58

print("Kart sayısı:", len(kartlar))
print("Tahmini kota:", len(kartlar) * GORSEL_BASI_NEURON, "/ 10000 neuron")
if len(kartlar) > MAKS_URETIM:
    print("Dikkat: kart sayısı sınırı aşıyor; bir çalıştırmada", MAKS_URETIM, "karttan sonra durulacak")

## Adım 3 — Araç ve sabırlı çizim

`uretim.py` bu klasörde duran yardımcı dosya. İçindeki `ciz`, Konu 8'de yazdığın fonksiyonla aynı işi yapar: yönerge ve tohum alır,
görselin baytlarını verir; başarısızsa `None`. Anahtarı `../anahtar.txt`'den kendisi okur. `gorsele_sor`'u Adım 7'de kullanacağız.
`requests`'i istek göndermek için değil, hata adını yazabilmek için alıyoruz (`requests.exceptions...`).

In [ ]:
import time
import requests
from pathlib import Path
from uretim import ciz, gorsele_sor

Toplu üretimde bir istek başarısız olursa bütün iş durmamalı. Konu 8'de hatayı yakalamıştık (Adım 8); bugün bir şey
ekliyoruz: **bekleyip yeniden denemek** (Konu 8'in Bonus 1'i bunun beklemesiz hâliydi).

- `requests.exceptions.RequestException`, `requests`'in bütün hatalarının ortak adı. Bağlantı kopması (`ConnectionError`) da,
  cevabın zamanında gelmemesi (`Timeout`) da bununla yakalanır.
- `time.sleep(5)` programı 5 saniye bekletir. Sunucu kısa süreli meşgulse ya da çok hızlı istek gönderdiysek (durum kodu 429) bu çoğu zaman yeter.
- En fazla üç deneme; son denemeden sonra beklemeye gerek yok (`if deneme < 3`).
- `return resim` fonksiyonu orada bitirir: başarılı olunca kalan denemeler yapılmaz.
- Günlük kota bittiyse beklemek işe yaramaz; kod bunu ayırt etmiyor. Üst üste `429` görürsen hücreyi durdur
  (üstteki menüden **Interrupt**); kota yenilenince kaldığın yerden devam edeceksin (Adım 4).

In [ ]:
def sabirli_ciz(yonerge, tohum):
    for deneme in [1, 2, 3]:
        try:
            resim = ciz(yonerge, tohum)
        except requests.exceptions.RequestException:
            print("Bağlanamadım ya da cevap gelmedi")
            resim = None
        if resim is not None:
            return resim
        if deneme < 3:
            print("Deneme", deneme, "olmadı, 5 saniye bekliyorum")
            time.sleep(5)
    return None

## Adım 4 — Üretim döngüsü: önce küçük bir deneme

Önce görsellerin klasörü:

In [ ]:
klasor = Path("izgara")
klasor.mkdir(exist_ok=True)

Döngüyü bir fonksiyona koyuyoruz, çünkü onu farklı sınırlarla iki kez çağıracağız: `toplu_uret(liste, sinir)` bir kart
listesi ve bir sınır alır. Her kart için:

- Dosya **zaten varsa** atla (`continue`: bu kartı bırak, sıradakine geç). Dosya varsa daha önce üretilmiştir; o yüzden durumu `uretildi`.
- **Sınıra** geldiysen dur (`break`: döngüyü bitir).
- Değilse **üret ve kaydet**; kartın içine sonucunu yaz (`kart["durum"]`).

Dosya adında tarih yok (Konu 8'deki `kaydet`'ten farkı): aynı kart her zaman aynı adı alır. Böylece "zaten var mı?" sorusu bir
**kaldığı yerden devam** düğmesine dönüşür. Konu 8'in `kaydet` fonksiyonunu kullanmamamızın sebebi de bu: o, her kayıtta yeni (tarihli)
bir ad üretiyordu.

In [ ]:
def toplu_uret(liste, sinir):
    denenen = 0
    for kart in liste:
        yol = klasor / kart["dosya"]
        if yol.exists():
            kart["durum"] = "uretildi"
            print("Zaten var, atlıyorum:", yol)
            continue
        if denenen >= sinir:
            print("Sınıra geldim, duruyorum")
            break
        resim = sabirli_ciz(kart["yonerge"], kart["tohum"])
        denenen = denenen + 1
        if resim is None:
            kart["durum"] = "olmadi"
            print("Olmadı, atlıyorum:", yol)
            continue
        with open(yol, "wb") as dosya:
            dosya.write(resim)
        kart["durum"] = "uretildi"
        print("Kaydedildi:", yol)
    print("Bu çalıştırmada denenen:", denenen)

Slaytta "önce küçük bir deneme" demiştik: yalnızca 3 görselle başlıyoruz.

In [ ]:
toplu_uret(kartlar, 3)

Üç görsel geldi ve döngü **sınırda durdu**. `izgara/` klasörüne bak: beğendin mi? Beğendiysen asıl sınırla devam.

**Tahmin et, sohbete yaz:** şimdi `toplu_uret(kartlar, MAKS_URETIM)` çalıştırırsak kaç istek gider?

In [ ]:
toplu_uret(kartlar, MAKS_URETIM)

İlk üç kart atlandı, kalan dokuzu üretildi: program **kaldığı yerden devam etti**. Bir dosyayı silip yeniden çalıştırırsan yalnızca o kart
üretilir. Kota biterse ya da internet koparsa, sonra aynı hücreyi çalıştırmak yeter.

Dikkat: adda olmayan bir şeyi (ör. `ozne`'yi ya da tohumu) değiştirip yeniden çalıştırırsan program bunun yeni bir iş olduğunu bilemez ve eski
görselleri atlar. Öyle bir değişiklikte yeni bir klasör aç.

## Adım 5 — Üretim günlüğü: kartları JSON'a yaz

Kartlar zaten günlüğün kendisi: her birinde stil, renk, dosya adı, yönergenin tamamı, tohum, model, brif ve sonuç (`durum`) var.
Hepsini bir **JSON** dosyasına yazıyoruz. JSON, Konu 2'de API'nin cevabındaki biçim: sözlükler ve listeler, metin olarak.

- `json.dump(kartlar, dosya, ...)`: önce yazılacak şey, sonra dosya.
- `ensure_ascii=False`: Türkçe harfler (`brif` alanındaki ç ve ş) `\u00e7` gibi kodlara dönüşmesin, olduğu gibi yazılsın.
- `indent=2`: her kart ayrı satırlarda, okunur biçimde.

In [ ]:
import json

with open("uretim_gunlugu.json", "w", encoding="utf-8") as dosya:
    json.dump(kartlar, dosya, ensure_ascii=False, indent=2)

print("Günlük yazıldı:", len(kartlar), "kart")

`uretim_gunlugu.json`'u VS Code'da aç. Program da onu geri okuyabilir; `json.load` dosyayı yeniden listeye çevirir:

In [ ]:
with open("uretim_gunlugu.json", encoding="utf-8") as dosya:
    okunan = json.load(dosya)

print(len(okunan))
print(okunan[5]["yonerge"])

`okunan[5]` altıncı kart (sıra 0'dan başlar). Bir müşteri aylar sonra "altıncı görselin aynısı" derse yönergesi ve tohumu elinde.

## Adım 6 — Kontak sayfası: bütün ızgara tek sayfada

Fotoğrafçılar filmdeki bütün kareleri tek kâğıda basıp en iyisini seçerdi: **kontak sayfası**. Bizimki bir ızgara:
satır sayısı stil sayısı, sütun sayısı renk sayısı. Konu 6'daki `subplot` döngüsü; bu kez birden çok satır var.
`subplot` kutuları soldan sağa doldurur, satır bitince alttakine geçer. Kartlar da stil stil sıralı olduğu için her satır bir stil olur.

- `plt.figure(figsize=(12, 9))`: sayfanın boyutu (genişlik, yükseklik; inç).
- `plt.imread` görseli dosyadan okur (Konu 8). Üretilemeyen kartın kutusu boş kalır.
- `plt.savefig` sayfayı dosyaya kaydeder; `bbox_inches="tight"` kenardaki boşluğu kırpar.

In [ ]:
import matplotlib.pyplot as plt

satir = len(stiller)
sutun = len(renkler)

plt.figure(figsize=(12, 9))
sira = 1
for kart in kartlar:
    yol = klasor / kart["dosya"]
    plt.subplot(satir, sutun, sira)
    if yol.exists():
        plt.imshow(plt.imread(yol))
    plt.title(kart["stil"] + " / " + kart["renk"], fontsize=9)
    plt.axis("off")
    sira = sira + 1

plt.savefig("kontak_sayfasi.png", bbox_inches="tight")
plt.show()

Sayfaya tasarımcı gözüyle bak: hangi stil brife uyuyor? Hangi renk? Bir satır ya da sütun öbürlerinden belirgin biçimde iyi ya da kötü mü?
Kazananını aşağıya yaz (dosya adıyla) ve sohbete tek cümle: **neden?**

In [ ]:
benim_secimim = "suluboya_adacayi.jpg"

## Adım 7 — Modele kendi çıktısını okutmak

Ölçek büyüyünce ilk elemeyi bir **görsel anlayan** modele yaptırabiliriz: metni ve görseli birlikte anlayan, **çok kipli** (multimodal)
bir model (Gemma). `gorsele_sor(yol, soru)` görseli ve soruyu ona gönderir, cevabı metin olarak verir.
Görsel yine base64 metni olarak taşınıyor, ama Konu 8'in tersi yönde: orada sunucu görseli base64'e çevirip bize göndermişti,
bu kez biz çevirip gönderiyoruz.

Önce tek bir görselle deneyelim (birkaç saniye sürer):

In [ ]:
ilk_yol = klasor / ilk_kart["dosya"]
cevap = gorsele_sor(ilk_yol, "Describe this image in one short sentence.")
print(cevap)

Şimdi her görsele aynı soruyu soruyoruz: brife 1 ile 5 arasında ne kadar uyuyor? Brifi İngilizce yazıyoruz (Konu 7).
Model bazen yalnızca bir sayı, bazen bir cümle yazar; o yüzden cevabı olduğu gibi kartın içine saklıyoruz. 12 istek birkaç dakika sürer.

In [ ]:
brif_ingilizce = "a poster for a quiet cafe where people come to study and work"
soru = f"On a scale of 1 to 5, how well does this image fit this brief: {brif_ingilizce}? Answer with only the number."

for kart in kartlar:
    yol = klasor / kart["dosya"]
    if yol.exists():
        kart["model_puani"] = gorsele_sor(yol, soru)
        print(kart["dosya"], kart["model_puani"])

print("Benim seçimim:", benim_secimim)

Modelin en yüksek puan verdikleri arasında senin seçimin var mı? Model brifteki kelimeleri görselde neyle eşliyor olabilir?

Model bir **ilk eleme** aracı, karar verici değil: onun da bir zevki, bir yanlılığı var (Konu 3). Kararı sen verirsin;
model yüz görseli ona indirmene yardım eder. Günlüğü puanlarla birlikte yeniden yazalım:

In [ ]:
with open("uretim_gunlugu.json", "w", encoding="utf-8") as dosya:
    json.dump(kartlar, dosya, ensure_ascii=False, indent=2)

print("Günlük güncellendi")

## Bonus

**1. Tohum ekseni.** Adım 6'da seçtiğin kazananı (`benim_secimim`) üç farklı tohumla üret. Önce kartını listede buluyoruz;
dosya adına tohumu da yazıyoruz ki her tohum ayrı bir dosya olsun.

In [ ]:
for kart in kartlar:
    if kart["dosya"] == benim_secimim:
        kazanan = kart

kazanan_stil = kazanan["stil"]
kazanan_renk = kazanan["renk"]

for tohum in [1, 2, 3]:
    yol = klasor / f"{kazanan_stil}_{kazanan_renk}_{tohum}.jpg"
    if not yol.exists():
        resim = sabirli_ciz(kazanan["yonerge"], tohum)
        if resim is not None:
            with open(yol, "wb") as dosya:
                dosya.write(resim)
    print(yol)

**2. Sınır gerçekten çalışıyor mu?** Isınma'daki `renkler` listesine ve Adım 1'deki `renk_yonergeleri` sözlüğüne bir palet ekle
(ör. `"mono"` ve `"black and white palette"`). Isınma'yı ve Adım 1'in hücrelerini yeniden çalıştır, sonra aşağıdaki hücreyi çalıştır.
Kaç yeni kart var, kaçı üretildi, neden?

In [ ]:
toplu_uret(kartlar, 2)